### Import Libraries

In [1]:
import time
import requests
from datasets import load_dataset
from google.oauth2 import service_account
from google.auth.transport.requests import AuthorizedSession
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

### Load dataset

In [2]:
ds = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42)
nombres = ds.features["label"].names

### Functions

In [3]:
URL = "https://crewia-classifier-judge-401145028214.us-central1.run.app"
JSON_CREDS = "proyects-execution.json"

In [4]:
creds = service_account.IDTokenCredentials.from_service_account_file(
    JSON_CREDS, target_audience=URL
)
session = AuthorizedSession(creds)

In [5]:
def predict_sentiment(text):
    """Return (output, error): exactly one of the two is None."""
    try:
        r = requests.post(URL, json={"review": text}, timeout=300)
    except requests.RequestException as e:
        # timeout, connection reset, DNS... (no HTTP response at all)
        return None, f"{type(e).__name__}: {e}"

    if not r.ok:
        # the API answers {"error": "..."}; Cloud Run itself (403, 429, 503, 504) answers HTML/plain text
        try:
            detail = r.json().get("error", r.text)
        except ValueError:
            detail = r.text.strip()[:500]
        return None, f"HTTP {r.status_code} {r.reason}: {detail}"

    return r.json(), None


In [6]:
def performance(name, y_true, y_pred):
    print(f"{name:<11}"
          f"accuracy={accuracy_score(y_true, y_pred):.4f}  "
          f"precision={precision_score(y_true, y_pred, zero_division=0):.4f}  "
          f"recall={recall_score(y_true, y_pred, zero_division=0):.4f}  "
          f"f1={f1_score(y_true, y_pred, zero_division=0):.4f}")

### Inference

In [7]:
N = 100
c = 10

classifier_pred = []
judge_pred = []
real_label = []
real_text = []

In [8]:
failed = []
start = time.perf_counter()
t_block = start


for i, ex in enumerate(ds.select(range(N)), start=1):

  text = ex['text']
  label = ex['label']

  output, error = predict_sentiment(text)
  if error is not None:
    failed.append((i - 1, error))
    print(f"Iteration {i} FAILED -> {error}")
    continue

  classifier_pred.append(output['classifier']['label'])
  judge_pred.append(output['judge']['final_label'])
  real_label.append(label)
  real_text.append(text)

  if i % c == 0:
    now = time.perf_counter()
    print(f"Iterations {i - c + 1}-{i}: {now - t_block:.2f} s "
          f"(elapsed: {now - start:.2f} s)")
    t_block = now

print(f"Total: {N} iterations in {time.perf_counter() - start:.2f} s "
      f"({len(failed)} failed)")

for idx, error in failed:
    print(f"[{idx}] {error}")

Iterations 1-10: 95.18 s (elapsed: 95.18 s)
Iterations 11-20: 83.84 s (elapsed: 179.01 s)
Iterations 21-30: 111.92 s (elapsed: 290.94 s)
Iterations 31-40: 83.35 s (elapsed: 374.28 s)
Iterations 41-50: 80.59 s (elapsed: 454.87 s)
Iterations 51-60: 76.56 s (elapsed: 531.44 s)
Iterations 61-70: 83.83 s (elapsed: 615.27 s)
Iterations 71-80: 84.39 s (elapsed: 699.66 s)
Iteration 89 FAILED -> HTTP 502 Bad Gateway: ValueError: Failed to validate structured output with model Classification: 1 validation error for Classification
  Invalid JSON: EOF while parsing a value at line 1 column 0 [type=json_invalid, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Iterations 81-90: 71.59 s (elapsed: 771.25 s)
Iterations 91-100: 80.57 s (elapsed: 851.81 s)
Total: 100 iterations in 851.81 s (1 failed)
[88] HTTP 502 Bad Gateway: ValueError: Failed to validate structured output with model Classification: 1 validation error for Classification


### Performance

In [9]:
to_int = {"negative": 0, "positive": 1}

In [10]:
clf = [to_int[p.strip().lower()] for p in classifier_pred]
judge = [to_int[p.strip().lower()] for p in judge_pred]

In [11]:
performance("Classifier", real_label, clf)

Classifier accuracy=0.9293  precision=0.9333  recall=0.9130  f1=0.9231


In [12]:
performance("Judge", real_label, judge)

Judge      accuracy=0.9394  precision=0.9348  recall=0.9348  f1=0.9348


### Cases where the judge improved the classifier

In [13]:
improved = [
    i for i, (y, c, j) in enumerate(zip(real_label, clf, judge))
    if c != y and j == y
]

print(f"Judge improved the classifier in {len(improved)} of {len(real_label)} cases")

for i in improved:
    print(f"\n[{i}] real={nombres[real_label[i]]} | "
          f"classifier={classifier_pred[i]} | judge={judge_pred[i]}")
    print(real_text[i][:300])

Judge improved the classifier in 1 of 99 cases

[29] real=pos | classifier=negative | judge=positive
It's good to see that Vintage Film Buff have correctly categorized their excellent DVD release as a "musical", for that's what this film is, pure and simple. Like its unofficial remake, Murder at the Windmill (1949), the murder plot is just an excuse for an elaborate girlie show with Kitty Carlisle 


### Cases where the judge worsened the classifier

In [14]:
worsened = [
    i for i, (y, c, j) in enumerate(zip(real_label, clf, judge))
    if c == y and j != y
]

print(f"Judge worsened the classifier in {len(worsened)} of {len(real_label)} cases")

for i in worsened:
    print(f"\n[{i}] real={nombres[real_label[i]]} | "
          f"classifier={classifier_pred[i]} | judge={judge_pred[i]}")
    print(real_text[i][:300])

print(f"\nNet effect of the judge: {len(improved) - len(worsened):+d} correct predictions")

Judge worsened the classifier in 0 of 99 cases

Net effect of the judge: +1 correct predictions


### Cases where both the classifier and the judge failed

In [15]:
both_failed = [
    i for i, (y, c, j) in enumerate(zip(real_label, clf, judge))
    if c != y and j != y
]

print(f"Both failed in {len(both_failed)} of {len(real_label)} cases")

for i in both_failed:
    print(f"\n[{i}] real={nombres[real_label[i]]} | "
          f"classifier={classifier_pred[i]} | judge={judge_pred[i]}")
    print(real_text[i][:300])

Both failed in 6 of 99 cases

[3] real=pos | classifier=negative | judge=negative
In the process of trying to establish the audiences' empathy with Jake Roedel (Tobey Maguire) the filmmakers slander the North and the Jayhawkers. Missouri never withdrew from the Union and the Union Army was not an invading force. The Southerners fought for State's Rights: the right to own slaves, 

[4] real=neg | classifier=positive | judge=positive
Yeh, I know -- you're quivering with excitement. Well, *The Secret Lives of Dentists* will not upset your expectations: it's solidly made but essentially unimaginative, truthful but dull. It concerns the story of a married couple who happen to be dentists and who share the same practice (already a r

[32] real=pos | classifier=negative | judge=negative
Gus Van Sant has made some excellent films. I truly am a fan.<br /><br />However, I can't help but feel that the cerebral edge of Tom Robbins book "Even Cowgirls Get the Blues" is lost in translation to the bi